<a href="https://colab.research.google.com/github/famub/2026-GP1-02/blob/main/mufsal_eff.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install kagglehub
import kagglehub
path = kagglehub.dataset_download("shashwatwork/knee-osteoarthritis-dataset-with-severity")
print("Path to dataset files:", path)

In [ ]:
!ls {path}

In [ ]:
from torchvision import transforms, datasets

base = path

tf = transforms.Compose([
    transforms.Grayscale(3),
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

train = datasets.ImageFolder(f"{base}/train", transform=tf)
val   = datasets.ImageFolder(f"{base}/val",   transform=tf)
test  = datasets.ImageFolder(f"{base}/test",  transform=tf)

print("train:", len(train), "| val:", len(val), "| test:", len(test))
print("classes:", train.classes)


In [ ]:
from collections import Counter

for name, ds in [("train", train), ("val", val), ("test", test)]:
    counts = Counter(ds.targets)
    print(name, {ds.classes[k]: v for k, v in sorted(counts.items())})


In [ ]:
import torch

counts = Counter(train.targets)
total = sum(counts.values())
weights = [total / (5 * counts[i]) for i in range(5)]
class_weights = torch.tensor(weights, dtype=torch.float)

print("class weights:", [round(w, 2) for w in weights])


In [ ]:
train_tf = transforms.Compose([
    transforms.Grayscale(3),
    transforms.Resize((224,224)),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

train = datasets.ImageFolder(f"{base}/train", transform=train_tf)
print("train:", len(train))


In [ ]:
from torch.utils.data import DataLoader

val = datasets.ImageFolder(f"{base}/val", transform=tf)
test = datasets.ImageFolder(f"{base}/test", transform=tf)

train_loader = DataLoader(train, batch_size=32, shuffle=True)
val_loader = DataLoader(val, batch_size=32, shuffle=False)
test_loader = DataLoader(test, batch_size=32, shuffle=False)

images, labels = next(iter(train_loader))
print("batch shape:", images.shape)
print("labels:", labels[:10])


In [ ]:
import cv2
import numpy as np
from PIL import Image
from torch.utils.data import DataLoader

# 1) دالة التنظيف: noise reduction + تحسين التباين
class Denoise:
    def __call__(self, img):
        g = np.array(img.convert("L"))                      # رمادي
        g = cv2.GaussianBlur(g, (3, 3), 0)                  # إزالة الضوضاء
        g = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(g)  # تحسين التباين
        return Image.fromarray(g)

# 2) pipeline للـ val و test (بدون augmentation)
tf = transforms.Compose([
    Denoise(),
    transforms.Grayscale(3),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# 3) pipeline للـ train (مع augmentation)
train_tf = transforms.Compose([
    Denoise(),
    transforms.Grayscale(3),
    transforms.Resize((224, 224)),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# 4) نعيد تحميل الداتا بالـ pipeline الجديد
train = datasets.ImageFolder(f"{base}/train", transform=train_tf)
val   = datasets.ImageFolder(f"{base}/val",   transform=tf)
test  = datasets.ImageFolder(f"{base}/test",  transform=tf)

train_loader = DataLoader(train, batch_size=32, shuffle=True)
val_loader   = DataLoader(val,   batch_size=32, shuffle=False)
test_loader  = DataLoader(test,  batch_size=32, shuffle=False)

images, labels = next(iter(train_loader))
print("batch shape:", images.shape)
print("train:", len(train), "| val:", len(val), "| test:", len(test))


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for k in range(5):
    path_k = train.samples[train.targets.index(k)][0]
    original = Image.open(path_k).convert("L")
    cleaned = Denoise()(original)
    axes[0][k].imshow(original, cmap="gray"); axes[0][k].set_title(f"Grade {k} - Original"); axes[0][k].axis("off")
    axes[1][k].imshow(cleaned, cmap="gray"); axes[1][k].set_title(f"Grade {k} - Cleaned"); axes[1][k].axis("off")
plt.show()

# مو مهم بس عشان اشوف الصور قبل وبعد

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

def crop_center(img, top=60, bottom=60):
    img = img.resize((224, 224))
    return img.crop((0, top, 224, 224 - bottom))   # 224 x 104

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for k in range(5):
    p = train.samples[train.targets.index(k)][0]
    original = Denoise()(Image.open(p).convert("L")).resize((224, 224))
    cropped = crop_center(Denoise()(Image.open(p).convert("L")))
    axes[0][k].imshow(original, cmap="gray"); axes[0][k].set_title(f"Grade {k} - Full"); axes[0][k].axis("off")
    axes[1][k].imshow(cropped, cmap="gray"); axes[1][k].set_title(f"Grade {k} - Cropped"); axes[1][k].axis("off")
plt.show()


In [ ]:
mkdir -p ~/.kaggle && echo KGAT_c95bdc246114faa95a33420354e152dd > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token

In [ ]:
!kaggle datasets download -d shashwatwork/knee-osteoarthritis-dataset-with-severity
!unzip -q knee-osteoarthritis-dataset-with-severity.zip -d mufsal_dataset


In [ ]:
import os
import cv2
import numpy as np
from PIL import Image
from collections import Counter
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
from tqdm import tqdm

# تحديد مسار الداتاسيت الأساسي (تأكدي أن المسار يطابق مكان وجود مجلداتك في كولاب)
base = "/content/drive/MyDrive/mufsal_dataset" # أو المسار لديك


In [ ]:
import os
import cv2
import numpy as np
from PIL import Image
from collections import Counter
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
from tqdm import tqdm

# --- التعديل هنا: استخدام مسار كولاب المباشر بعد سحب الداتاسيت من كاجل ---
base = "/content/mufsal_dataset"

# 1) دالة التنظيف وإزالة الضوضاء وتحسين التباين (CLAHE)
class Denoise:
    def __call__(self, img):
        g = np.array(img.convert("L"))
        g = cv2.GaussianBlur(g, (3, 3), 0)
        g = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(g)
        return Image.fromarray(g)

# 2) خط التحويلات للـ Validation و Test (بدون عشوائية)
tf = transforms.Compose([
    Denoise(),
    transforms.Grayscale(3),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# 3) خط التحويلات للـ Train (مع إضافة Augmentation)
train_tf = transforms.Compose([
    Denoise(),
    transforms.Grayscale(3),
    transforms.Resize((224, 224)),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# 4) تحميل المجلدات بناءً على المسار الجديد
train = datasets.ImageFolder(f"{base}/train", transform=train_tf)
val = datasets.ImageFolder(f"{base}/val", transform=tf)
test = datasets.ImageFolder(f"{base}/test", transform=tf)

# 5) تجهيز الدفعات (DataLoaders)
train_loader = DataLoader(train, batch_size=32, shuffle=True)
val_loader = DataLoader(val, batch_size=32, shuffle=False)
test_loader = DataLoader(test, batch_size=32, shuffle=False)

print(f"Data Loaded Successfully! Train: {len(train)} | Val: {len(val)} | Test: {len(test)}")


In [ ]:
from collections import Counter
from torchvision import models

# 1. حساب أوزان الفئات لمعالجة عدم توازن البيانات
counts = Counter(train.targets)
total = sum(counts.values())
weights = [total / (5 * counts[i]) for i in range(5)]
class_weights = torch.tensor(weights, dtype=torch.float)
print("Class Weights Calculated:", [round(w, 2) for w in weights])

# 2. إعداد الجهاز للعمل على كرت الشاشة (GPU) لسرعة التدريب
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 3. تحميل نموذج EfficientNet-B0 وتعديل الطبقة الأخيرة لتناسب 5 فئات (من 0 إلى 4)
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
num_ftrs = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_ftrs, 5)
model = model.to(device)

# 4. دالة الخسارة مع أوزان الفئات والمحسّن (مع تعديل معدل التعلم لرفع الدقة)
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5)

# 5. دالة التدريب والتحقق
def train_and_validate(model, train_loader, val_loader, criterion, optimizer, scheduler, epochs=25):
    best_acc = 0.0

    for epoch in range(epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")
        print("-" * 30)

        # مرحلة التدريب
        model.train()
        train_loss, train_corrects, total_train = 0.0, 0, 0

        for inputs, labels in tqdm(train_loader, desc="Training"):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * inputs.size(0)
            train_corrects += torch.sum(preds == labels.data)
            total_train += labels.size(0)

        epoch_train_loss = train_loss / total_train
        epoch_train_acc = (train_corrects.double() / total_train).item()

        # مرحلة التحقق (Validation)
        model.eval()
        val_loss, val_corrects, total_val = 0.0, 0, 0

        with torch.no_grad():
            for inputs, labels in tqdm(val_loader, desc="Validation"):
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                _, preds = torch.max(outputs, 1)

                val_loss += loss.item() * inputs.size(0)
                val_corrects += torch.sum(preds == labels.data)
                total_val += labels.size(0)

        epoch_val_loss = val_loss / total_val
        epoch_val_acc = (val_corrects.double() / total_val).item()

        scheduler.step(epoch_val_loss)

        print(f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc*100:.2f}%")
        print(f"Val Loss:   {epoch_val_loss:.4f} | Val Acc:   {epoch_val_acc*100:.2f}%")

        # حفظ أفضل موديل تلقائياً
        if epoch_val_acc > best_acc:
            best_acc = epoch_val_acc
            torch.save(model.state_dict(), 'best_mufsal_model.pth')
            print(f"--> Saved Best Model with Val Acc: {best_acc*100:.2f}%")

    print(f"\nTraining Complete. Best Validation Accuracy: {best_acc*100:.2f}%")

# بدء التدريب لـ 25 دورة لتحقيق أفضل دقة ممكنة
train_and_validate(model, train_loader, val_loader, criterion, optimizer, scheduler, epochs=25)


Class Weights Calculated: [0.51, 1.1, 0.76, 1.53, 6.68]
Using device: cpu
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 129MB/s] 



Epoch 1/25
------------------------------


Validation: 100%|██████████| 26/26 [00:41<00:00,  1.61s/it]


Train Loss: 1.3773 | Train Acc: 37.38%
Val Loss:   1.1065 | Val Acc:   48.31%
--> Saved Best Model with Val Acc: 48.31%

Epoch 2/25
------------------------------


...

Validation: 100%|██████████| 26/26 [00:40<00:00,  1.56s/it]


Train Loss: 0.6995 | Train Acc: 64.40%
Val Loss:   0.8657 | Val Acc:   61.99%
--> Saved Best Model with Val Acc: 61.99%

Epoch 6/25
------------------------------


Validation: 100%|██████████| 26/26 [00:40<00:00,  1.56s/it]


Train Loss: 0.6493 | Train Acc: 66.68%
Val Loss:   0.8873 | Val Acc:   59.08%

Epoch 7/25
------------------------------


Training:  95%|█████████▌| 172/181 [21:01<01:05,  7.25s/it]

In [ ]:
import shutil
# نسخ ملف الموديل المحفوظ إلى مجلد الدرايف لديكِ
shutil.copy('best_mufsal_model.pth', '/content/drive/MyDrive/best_mufsal_model.pth')
print("Model saved to Google Drive successfully!")
